# Dados para atualizar o dashboard


In [ ]:
from pyspark.sql import SparkSession


spark = (
    SparkSession
    .builder
    .appName("Agendamentos")
    .config("spark.jars", "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "miniominio")
    .config("spark.hadoop.fs.s3a.secret.key", "miniominio")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)


In [7]:
data_inicial = "20/07/2026"
data_final = "19/08/2026"

data_inicial_read = data_inicial.replace("/", "-")
data_final_read = data_final.replace("/", "-")

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(f"s3a://agendamentos/gold/agendamentos_{data_inicial_read}_{data_final_read}")
)

df.createOrReplaceTempView("agendamentos")


# Apresentacao faltas(2)

In [8]:
spark.sql("""
    SELECT
        cidade,
        ano,
        competencia,
        SUM(CASE WHEN situacao = 'VALIDADO' THEN 1 ELSE 0 END) AS atendimento,
        SUM(CASE WHEN situacao = 'FALTA' THEN 1 ELSE 0 END) AS falta,
        '{indice}' AS indice,
        concat(ano2, ' - ', competencia) AS competencia,
        comp
    FROM agendamentos
    GROUP BY
        cidade,
        ano,
        ano2,
        competencia,
        comp
    ORDER BY
        ano,
        cidade;
""").show()

+-----------+----+--------------------+-----------+-----+--------+--------------------+-------------+
|     cidade| ano|         competencia|atendimento|falta|  indice|         competencia|         comp|
+-----------+----+--------------------+-----------+-----+--------+--------------------+-------------+
|    Itaguaí|2026|Competencia 20/07...|       3645| 1256|{indice}|26 - Competencia ...|20/07 a 19/08|
|     Japeri|2026|Competencia 20/07...|        330|  110|{indice}|26 - Competencia ...|20/07 a 19/08|
|   Mesquita|2026|Competencia 20/07...|       1556|  192|{indice}|26 - Competencia ...|20/07 a 19/08|
|  Nilópolis|2026|Competencia 20/07...|       1839|   92|{indice}|26 - Competencia ...|20/07 a 19/08|
|Nova Iguaçu|2026|Competencia 20/07...|       1071|  140|{indice}|26 - Competencia ...|20/07 a 19/08|
|  Paracambi|2026|Competencia 20/07...|       1639|  335|{indice}|26 - Competencia ...|20/07 a 19/08|
|  Queimados|2026|Competencia 20/07...|       1183|  394|{indice}|26 - Competencia

# Apresentacao (2022)

In [9]:
spark.sql("""
    SELECT
        concat('{indice} ', comp, " ", ano) AS competencia,
        '{indice}' as indice,
        concat(comp, " ", ano) as competencia,
        count(*) as total
    FROM agendamentos
    where situacao = 'VALIDADO'
    GROUP BY
        competencia,
        comp,
        ano
        
    ORDER BY
        competencia
""").show()

+--------------------+--------+------------------+-----+
|         competencia|  indice|       competencia|total|
+--------------------+--------+------------------+-----+
|{indice} 20/07 a ...|{indice}|20/07 a 19/08 2026|11263|
+--------------------+--------+------------------+-----+



# RELATORIO GERAL

In [12]:

spark.sql("""
    WITH dados AS (
        SELECT
            CASE
                WHEN procedimento IN (
                    'LOGOAUDIOMETRIA (LDV-IRF-LRF)',
                    'AUDIOMETRIA TONAL LIMIAR (VIA AEREA / OSSEA) IMPEDANCIOMETRIA E LOGOAUDIOMETRIA'
                )
                THEN 'AUDIOMETRIA TONAL LIMIAR IMPED e LOGOAUDOMETRIA (LDV-IRF-LRF)'
    
                WHEN procedimento IN (
                    'CORE BIOPSIA MAMA POR ULTRASSONOGRAFIA (INCLUI RESULTADO DO HISTOPATOLOGICO - 4 FRAGMENTOS)'
                )
                THEN 'BIÓPSIA MAMA'
    
                WHEN procedimento IN (
                    'BIOPSIA PROSTATA TRANSRETAL GUIADA POR ULTRASSONOGRAFIA- (ATE 8 FRAGMENTOS) (INCLUI RESULTADO DO HISTOPATOLOGICO)'
                )
                THEN 'BIÓPSIA PRÓSTATA'
                
                WHEN procedimento IN (
                    'FACOEMULSIFICACAO C/ IMPLANTE DE LENTE INTRA-OCULAR DOBRAVEL',
                    'CAPSULOTOMIA A YAG LASER - MONOCULAR',
                    'TRATAMENTO CIRURGICO DE PTERIGIO',
                    'EXERESE DE CALAZIO E OUTRAS PEQUENAS LESOES DA PALPEBRA E SUPERCILIOS'
                )
                THEN 'CIRURGIA OFTALMOLOGICA'
    
                WHEN procedimento IN (
                    'COLPOSCOPIA COM BIOPSIA E RESULTADO DA BIOPSIA',
                    'COLPOSCOPIA'
                )
                THEN 'COLPOSCOPIA COM BIOPSIA E RESULTADO DA BIOPSIA'
    
                WHEN procedimento IN (
                    'CONSULTA - CARDIOLOGIA COM ELETROCARDIOGRAMA'
                )
                THEN 'CONSULTAS CARDIO C/ ELETRO'
    
                WHEN procedimento IN (
                    'CONSULTA -  OFTALMOLOGISTA COM TONOMETRIA E FUNDOSCOPIA',
                    'CONSULTA - OFTALMOLOGISTA'
                )
                THEN 'CONSULTAS OFTALMOLOGIA'
    
                WHEN procedimento IN (
                    'CONSULTA - UROLOGISTA',
                    'CONSULTA - REUMATOLOGISTA',
                    'CONSULTA - PSIQUIATRA - PEDIATRICO',
                    'CONSULTA - PSIQUIATRA',
                    'CONSULTA - PROCTOLOGISTA',
                    'CONSULTA - PNEUMOLOGISTA',
                    'CONSULTA - PNEUMOLOGISTA -PEDIATRICO',
                    'CONSULTA - ORTOPEDISTA E TRAUMATOLOGISTA',
                    'CONSULTA - OTORRINOLARINGOLOGISTA',
                    'CONSULTA - DERMATOLOGISTA',
                    'CONSULTA - ENDOCRINOLOGISTA e METABOLISTA',
                    'CONSULTA - ENDOCRINOLOGISTA e METABOLISTA - PEDIATRICO',
                    'CONSULTA - GASTROENTEROLOGISTA',
                    'CONSULTA - GASTROENTEROLOGISTA - PEDIATRICO',
                    'CONSULTA - GERIATRA',
                    'CONSULTA - GINECOLOGIA',
                    'CONSULTA - OBSTÉTRICA',
                    'CONSULTA - HEMATOLOGISTA',
                    'CONSULTA - HEMATOLOGISTA - PEDIATRICO',
                    'CONSULTA - INFECTOLOGISTA',
                    'CONSULTA - MASTOLOGISTA',
                    'CONSULTA - NEFROLOGISTA',
                    'CONSULTA - NEUROCIRURGIÃO',
                    'CONSULTA - NEUROLOGISTA',
                    'CONSULTA - NEUROLOGISTA PEDIATRICO',
                    'CONSULTA - CIRURGIAO GERAL',
                    'CONSULTA - PEDIATRIA',
                    'CONSULTA - CLINICO',
                    'CONSULTA - CARDIOLOGISTA',
                    'CONSULTA - CARDIOLOGISTA - PEDIATRICO',
                    'CONSULTA - ANGIOLOGISTA',
                    'CONSULTA - ALERGISTA e IMUNOLOGISTA'
                )
                THEN 'CONSULTAS ESPECIALIZADAS'
    
                WHEN procedimento IN (
                    'DENSITOMETRIA OSSEA DUO-ENERGETICA DE COLUNA (VERTEBRAS LOMBARES E/OU FEMUR)'
                )
                THEN 'DESINTOMETRIA'
    
                WHEN procedimento IN (
                    'ECOCARDIOGRAFIA TRANSTORACICA - PEDIATRICO (0 A 13 ANOS)',
                    'ECOCARDIOGRAFIA TRANSTORACICA'
                )
                THEN 'ECOCARDIOGRAFIA TRANSTORACICA'
    
                WHEN procedimento IN (
                    'ELETROENCEFALOGRAMA EM SONO INDUZIDO C/ OU S/ MEDICAMENTO (EEG)',
                    'ELETROENCEFALOGRAMA EM VIGILIA E SONO ESPONTANEO C/ OU S/ FOTOESTIMULO (EEG)',
                    'ELETROENCEFALOGRAMA QUANTITATIVO C/ MAPEAMENTO (EEG)'
                )
                THEN 'ELETROENCEFALOGRAMA'
    
                WHEN procedimento IN (
                    'ELETRONEUROMIOGRAMA (ENMG) CADA MEMBRO'
                )
                THEN 'ELETRONEUROMIOGRAFIA'
    
                WHEN procedimento IN (
                    'COLONOSCOPIA (COLOSCOPIA) SEM BIOPSIA',
                    'COLONOSCOPIA COM BIOPSIA E POLIPECTOMIA - INCLUINDO O RESULTADO DO HISTOPATOLOGICO',
                    'COLONOSCOPIA COM BIOPSIA - INCLUINDO RESULTADO DO HISTOPATOLOGICO',
                    'ESOFAGOGASTRODUODENOSCOPIA COM BIÓPSIAS INCLUINDO RESULTADO DO HISTOPATOLÓGICO',
                    'ESOFAGOGASTRODUODENOSCOPIA - SEM BIÓPSIA',
                    'ESOFAGOGASTRODUODENOSCOPIA COM BIÓPSIAS E POLIPECTOMIA INCLUINDO RESULTADO DO HISTOPATOLÓGICO'
                )
                THEN 'ENDOSCOPIAS DIGESTIVAS (ALTA E BAIXA)'
    
                WHEN procedimento IN (
                    'ESPIROMETRIA OU PROVA DE FUNCAO PULMONAR COMPLETA COM BRONCODILATADOR'
                )
                THEN 'ESPIROMETRIA'
    
                WHEN procedimento IN (
                    'DOSAGEM DE ACIDO URICO',
                    'DOSAGEM DE ACIDO VANILMANDELICO',
                    'DOSAGEM DE ALDOLASE',
                    'DOSAGEM DE AMILASE',
                    'DOSAGEM DE AMONIA',
                    'DOSAGEM DE BILIRRUBINA TOTAL E FRACOES',
                    'DOSAGEM DE CALCIO',
                    'DOSAGEM DE CALCIO IONIZAVEL',
                    'DOSAGEM DE CLORETO',
                    'DOSAGEM DE COLESTEROL HDL',
                    'DOSAGEM DE COLESTEROL LDL',
                    'DOSAGEM DE COLESTEROL TOTAL',
                    'DOSAGEM DE COLINESTERASE',
                    'DOSAGEM DE CREATININA',
                    'DOSAGEM DE CREATINOFOSFOQUINASE (CPK)',
                    'DOSAGEM DE CREATINOFOSFOQUINASE FRACAO MB',
                    'DOSAGEM DE DESIDROGENASE LATICA',
                    'DOSAGEM DE FERRITINA',
                    'DOSAGEM DE FERRO SERICO',
                    'DOSAGEM DE FOSFATASE ALCALINA',
                    'DOSAGEM DE GALACTOSE',
                    'DOSAGEM DE GAMA-GLUTAMIL-TRANSFERASE (GAMA GT)',
                    'DOSAGEM DE GLICOSE',
                    'DOSAGEM DE GLICOSE-6-FOSFATO DESIDROGENASE',
                    'DOSAGEM DE HEMOGLOBINA GLICOSILADA',
                    'DOSAGEM DE LIPASE',
                    'DOSAGEM DE MAGNESIO',
                    'DOSAGEM DE POTASSIO',
                    'DOSAGEM DE PROTEINAS TOTAIS',
                    'DOSAGEM DE PROTEINAS TOTAIS E FRACOES',
                    'DOSAGEM DE SODIO',
                    'DOSAGEM DE TRANSAMINASE GLUTAMICO-OXALACETICA (TGO)',
                    'DOSAGEM DE TRANSAMINASE GLUTAMICO-PIRUVICA (TGP)',
                    'DOSAGEM DE TRANSFERRINA',
                    'DOSAGEM DE TRIGLICERIDEOS',
                    'DOSAGEM DE UREIA',
                    'DOSAGEM DE VITAMINA B12',
                    'DOSAGEM DE 25 HIDROXIVITAMINA D',
                    'DOSAGEM DE FATOR IX',
                    'DOSAGEM DE FATOR V',
                    'DOSAGEM DE FATOR VII',
                    'DOSAGEM DE FATOR VIII',
                    'DOSAGEM DE FATOR VIII (INIBIDOR)',
                    'DOSAGEM DE FATOR VON WILLEBRAND (ANTIGENO)',
                    'DOSAGEM DE FATOR X',
                    'DOSAGEM DE FATOR XI',
                    'DOSAGEM DE FATOR XII',
                    'DOSAGEM DE FATOR XIII',
                    'DOSAGEM DE FIBRINOGENIO',
                    'DOSAGEM DE HEMOGLOBINA',
                    'DOSAGEM DE HEMOGLOBINA FETAL',
                    'DOSAGEM DE PROTEINA C FUNCIONAL',
                    'DOSAGEM DE PROTEINA S FUNCIONAL',
                    'DOSAGEM DE ALFA-FETOPROTEINA',
                    'DOSAGEM DE ANTIGENO PROSTATICO ESPECIFICO (PSA)',
                    'DOSAGEM DE COMPLEMENTO C3',
                    'DOSAGEM DE COMPLEMENTO C4',
                    'DOSAGEM DE IMUNOGLOBULINA A (IGA)',
                    'DOSAGEM DE IMUNOGLOBULINA E (IGE)',
                    'DOSAGEM DE IMUNOGLOBULINA M (IGM)',
                    'DOSAGEM DE PROTEINA C REATIVA',
                    'DOSAGEM DA FRAÇÃO C1Q DO COMPLEMENTO',
                    'DOSAGEM DO ANTÍGENO CA 125',
                    'DOSAGEM DE ANTI-BETA-2-GLICOPROTEÍNA I - IGM',
                    'DOSAGEM DE PROTEINAS (URINA DE 24 HORAS)',
                    'DOSAGEM DE 17-ALFA-HIDROXIPROGESTERONA',
                    'DOSAGEM DE 17-HIDROXICORTICOSTEROIDES',
                    'DOSAGEM DE ACIDO 5-HIDROXI-INDOL-ACETICO (SEROTONINA)',
                    'DOSAGEM DE ADRENOCORTICOTROFICO (ACTH)',
                    'DOSAGEM DE ANDROSTENEDIONA',
                    'DOSAGEM DE CALCITONINA',
                    'DOSAGEM DE CORTISOL',
                    'DOSAGEM DE DEHIDROEPIANDROSTERONA (DHEA)',
                    'DOSAGEM DE DIHIDROTESTOTERONA (DHT)',
                    'DOSAGEM DE ESTRADIOL',
                    'DOSAGEM DE ESTRIOL',
                    'DOSAGEM DE ESTRONA',
                    'DOSAGEM DE GASTRINA',
                    'DOSAGEM DE GLOBULINA TRANSPORTADORA DE TIROXINA',
                    'DOSAGEM DE GONADOTROFINA CORIONICA HUMANA (HCG, BETA HCG)',
                    'DOSAGEM DE HORMONIO DE CRESCIMENTO (HGH)',
                    'DOSAGEM DE HORMONIO FOLICULO-ESTIMULANTE (FSH)',
                    'DOSAGEM DE HORMONIO LUTEINIZANTE (LH)',
                    'DOSAGEM DE HORMONIO TIREOESTIMULANTE (TSH)',
                    'DOSAGEM DE INSULINA',
                    'DOSAGEM DE PARATORMONIO',
                    'DOSAGEM DE PROGESTERONA',
                    'DOSAGEM DE PROLACTINA',
                    'DOSAGEM DE SOMATOMEDINA C (IGF1)',
                    'DOSAGEM DE SULFATO DE HIDROEPIANDROSTERONA (DHEAS)',
                    'DOSAGEM DE TESTOSTERONA',
                    'DOSAGEM DE TESTOSTERONA LIVRE',
                    'DOSAGEM DE TIREOGLOBULINA',
                    'DOSAGEM DE TIROXINA (T4)',
                    'DOSAGEM DE TIROXINA LIVRE (T4 LIVRE)',
                    'DOSAGEM DE TRIIODOTIRONINA (T3)',
                    'DOSAGEM DE ACIDO MANDELICO',
                    'DOSAGEM DE ACIDO VALPROICO',
                    'DOSAGEM DE BARBITURATOS',
                    'DOSAGEM DE BENZODIAZEPINICOS',
                    'DOSAGEM DE CADMIO',
                    'DOSAGEM DE CARBAMAZEPINA',
                    'DOSAGEM DE COBRE',
                    'DOSAGEM DE FENITOINA',
                    'DOSAGEM DE LITIO',
                    'DOSAGEM DE ZINCO',
                    'DOSAGEM DE FRUTOSE',
                    'DOSAGEM DE FENILALANINA (CONTROLE / DIAGNOSTICO TARDIO)',
                    'DOSAGEM DE 17 HIDROXI PROGESTERONA EM PAPEL DE FILTRO (COMPONENTE DO TESTE DO PEZINHO)',
                    'DETERMINAÇÃO DE TEMPO DE COAGULAÇÃO',
                    'DETERMINAÇÃO DE TEMPO DE SANGRAMENTO -DUKE',
                    'DETERMINAÇÃO DE TEMPO DE SANGRAMENTO DE IVY',
                    'DETERMINAÇÃO DE TEMPO DE TROMBINA',
                    'DETERMINAÇÃO DE TEMPO DE TROMBOPLASTINA PARCIAL ATIVADA (TTP ATIVADA)',
                    'DETERMINAÇÃO DE TEMPO E ATIVIDADE DA PROTROMBINA (TAP)',
                    'DETERMINAÇÃO DE VELOCIDADE DE HEMOSSEDIMENTAÇÃO (VHS)',
                    'DETERMINAÇÃO DE COMPLEMENTO (CH50)',
                    'DETERMINAÇÃO DE FATOR REUMATOIDE',
                    'DETERMINAÇÃO QUANTITATIVA DE PROTEÍNA C REATIVA',
                    'DETERMINAÇÃO DE ÍNDICE DE TIROXINA LIVRE',
                    'DETERMINAÇÃO DE T3 REVERSO',
                    'PESQUISA DE CELULAS LE',
                    'PESQUISA DE CORPUSCULOS DE HEINZ',
                    'PESQUISA DE HEMOGLOBINA S',
                    'PESQUISA DE ANTICOAGULANTE LÚPICO',
                    'PESQUISA DE ANTICORPOS ANTI-HIV-1 (WESTERN BLOT)',
                    'PESQUISA DE ANTICORPOS ANTI-HIV-1 + HIV-2 (ELISA)',
                    'PESQUISA DE ANTICORPOS ANTI-HTLV-1 + HTLV-2',
                    'PESQUISA DE ANTICORPOS ANTI-RIBONUCLEOPROTEINA (RNP)',
                    'PESQUISA DE ANTICORPOS ANTI-SS-A (RO)',
                    'PESQUISA DE ANTICORPOS ANTI-SS-B (LA)',
                    'PESQUISA DE ANTICORPOS ANTIESTREPTOLISINA O (ASLO)',
                    'PESQUISA DE ANTICORPOS ANTITIREOGLOBULINA',
                    'PESQUISA DE ANTICORPOS CONTRA ANTIGENO DE SUPERFICIE DO VIRUS DA HEPATITE B (ANTI-HBS)',
                    'PESQUISA DE ANTICORPOS CONTRA ANTIGENO E DO VIRUS DA HEPATITE B (ANTI-HBE)',
                    'PESQUISA DE ANTICORPOS CONTRA O VIRUS DA HEPATITE C (ANTI-HCV)',
                    'PESQUISA DE ANTICORPOS CONTRA O VIRUS DO SARAMPO',
                    'PESQUISA DE ANTICORPOS HETEROFILOS CONTA O VIRUS EPSTEIN-BARR',
                    'PESQUISA DE ANTICORPOS IGG ANTICITOMEGALOVIRUS',
                    'PESQUISA DE ANTICORPOS IGG ANTITOXOPLASMA',
                    'PESQUISA DE ANTICORPOS IGG E IGM CONTRA ANTIGENO CENTRAL DO VIRUS DA HEPATITE B (ANTI-HBC-TOTAL)',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS DA HEPATITE A (HAV-IGG)',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS DA RUBEOLA',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS DA VARICELA-HERPES ZOSTER',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS EPSTEIN-BARR',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS HERPES SIMPLES',
                    'PESQUISA DE ANTICORPOS IGM CONTRA ANTIGENO CENTRAL DO VIRUS DA HEPATITE B (ANTI-HBC-IGM)',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS DA HEPATITE A (HAV-IGG)',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS DA RUBEOLA',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS DA VARICELA-HERPES ZOSTER',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS EPSTEIN-BARR',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS HERPES SIMPLES',
                    'PESQUISA DE ANTIGENO CARCINOEMBRIONARIO (CEA)',
                    'PESQUISA DE ANTIGENO DE SUPERFICIE DO VIRUS DA HEPATITE B (HBSAG)',
                    'PESQUISA DE ANTIGENO E DO VIRUS DA HEPATITE B (HBEAG)',
                    'PESQUISA DE CRIOGLOBULINAS',
                    'PESQUISA DE FATOR REUMATOIDE (WAALER-ROSE)',
                    'PESQUISA DE IMUNOGLOBULINA E (IGE) ALERGENO-ESPECIFICA',
                    'PESQUISA DE ANTICORPOS ANTICORPOS ANTI-HTLV-1 (WESTERN-BLOT)',
                    'PESQUISA DE EOSINOFILOS',
                    'PESQUISA DE GORDURA FECAL',
                    'PESQUISA DE LARVAS NAS FEZES',
                    'PESQUISA DE LEUCOCITOS NAS FEZES',
                    'PESQUISA DE LEVEDURAS NAS FEZES',
                    'PESQUISA DE OVOS E CISTOS DE PARASITAS',
                    'PESQUISA DE SANGUE OCULTO NAS FEZES',
                    'PESQUISA DE TREPONEMA PALLIDUM',
                    'PESQUISA DE ESPERMATOZOIDES (APOS VASECTOMIA)',
                    'PROVA DO LATEX P/ PESQUISA DO FATOR REUMATOIDE',
                    'PESQUISA DE FATOR RH (INCLUI D FRACO)',
                    'DETERMINACAO DE CAPACIDADE DE FIXACAO DO FERRO',
                    'DETERMINACAO DE CURVA GLICEMICA CLASSICA (5 DOSAGENS)',
                    'DETERMINACAO DE ANTICORPOS ANTIPLAQUETARIOS',
                    'DETERMINACAO DIRETA E REVERSA DE GRUPO ABO',
                    'BACILOSCOPIA DIRETA P/ BAAR TUBERCULOSE (DIAGNÓSTICA)',
                    'BACILOSCOPIA DIRETA P/ BAAR (HANSENIASE)',
                    'BACILOSCOPIA DIRETA P/ BAAR TUBERCULOS (CONTROLE)',
                    'TESTE DIRETO DE ANTIGLOBULINA HUMANA (TAD)',
                    'ANTIBETA 2 GLICOPROTEINA I -IGG',
                    'ANTIBIOGRAMA',
                    'ANTIBIOGRAMA P/ MICOBACTERIAS',
                    'TESTE INDIRETO DE ANTIGLOBULINA HUMANA (TIA)',
                    'EXAME ANATOMO-PATOLOGICO DO COLO UTERINO - PECA CIRURGICA',
                    'EXAME ANATOMO-PATOLÓGICO PARA CONGELAMENTO / PARAFINA POR PEÇA CIRURGICA OU POR BIOPSIA (EXCETO COLO UTERINO E MAMA)',
                    'EXAME ANATOMOPATOLOGICO DE MAMA - BIOPSIA',
                    'EXAME ANATOMOPATOLOGICO DE MAMA - PECA CIRURGICA',
                    'EXAME ANATOMO-PATOLOGICO DO COLO UTERINO - BIOPSIA',
                    'CONTAGEM DE PLAQUETAS',
                    'CONTAGEM DE RETICULOCITOS',
                    'CONTAGEM DE LINFOCITOS B',
                    'CONTAGEM DE LINFOCITOS CD4/CD8',
                    'CONTAGEM DE LINFOCITOS T TOTAIS',
                    'TESTE DE TOLERANCIA A INSULINA / HIPOGLICEMIANTES ORAIS',
                    'ELETROFORESE DE HEMOGLOBINA',
                    'ERITROGRAMA (ERITROCITOS, HEMOGLOBINA, HEMATOCRITO)',
                    'HEMATOCRITO',
                    'HEMOGRAMA COMPLETO',
                    'LEUCOGRAMA',
                    'PROVA DE RETRACAO DO COAGULO',
                    'PROVA DO LAÇO',
                    'DETECÇÃO DE ÁCIDOS NUCLEICOS DO HIV-1 (QUALITATIVO)',
                    'DETECÇÃO DE RNA DO VÍRUS DA HEPATITE C (QUALITATIVO)',
                    'DETECÇÃO DE RNA DO HTLV-1',
                    'EXAME COPROLOGICO FUNCIONAL',
                    'ANALISE DE CARACTERES FISICOS, ELEMENTOS E SEDIMENTO DA URINA',
                    'CLEARANCE DE CREATININA',
                    'CLEARANCE DE UREIA',
                    'TESTE P/ INVESTIGACAO DO DIABETES INSIPIDUS',
                    'BACTERIOSCOPIA (GRAM)',
                    'CULTURA DE BACTERIAS P/ IDENTIFICACAO',
                    'CULTURA PARA BAAR',
                    'HEMOCULTURA',
                    'EXAME CITOPATOLOGICO CERVICO-VAGINAL/MICROFLORA- Registro no SISCOLO',
                    'EXAME CITOPATOLÓGICO CERVICO-VAGINAL/MICROFLORA',
                    'IMUNOHISTOQUIMICA DE NEOPLASIAS MALIGNAS (POR MARCADOR)',
                    'ELETROFORESE DE PROTEINAS'
                )
                THEN 'EXAMES LABORATORIAIS'
    
                WHEN procedimento IN (
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE NO PRÉ/PÓS CIRURGIAS UROGINECOLÓGICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES C/ DISFUNÇÕES UROGINECOLÓGICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE ONCOLÓGICO CLÍNICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE NO PRÉ E PÓS CIRURGIA ONCOLÓGICA',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM ALTERAÇÕES OCULOMOTORAS CENTRAIS C/ COMPROMETIMENTO SISTÊMICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM ALTERAÇÕES OCULOMOTORAS CENTRAIS C/ COMPROMETIMENTO SISTÊMICO - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM ALTERAÇÕES OCULOMOTORAS PERIFÉRICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM ALTERAÇÕES OCULOMOTORAS PERIFÉRICAS - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO RESPIRATÓRIO COM COMPLICAÇÕES SISTÊMICAS - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO RESPIRATÓRIO COM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO RESPIRATÓRIO SEM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO CLÍNICO CARDIOVASCULAR',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE PRÉ/PÓS CIRURGIA CARDIOVASCULAR',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS DISFUNÇÕES VASCULARES PERIFÉRICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES NO PRÉ E PÓS-OPERATÓRIO NAS DISFUNÇÕES MÚSCULO ESQUELÉTICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS ALTERAÇÕES MOTORAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES NO PRÉ E PÓS-OPERATÓRIO NAS DISFUNÇÕES MUSCULO- ESQUELETICAS C/ COMPLICAÇÕES SISTEMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM DISTÚRBIOS NEURO-CINÉTICO-FUNCIONAIS SEM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM DISTÚRBIOS NEURO-CINÉTICO-FUNCIONAIS COM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM DISTÚRBIOS NEURO-CINÉTICO-FUNCIONAIS COM COMPLICAÇÕES SISTÊMICAS - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS DESORDENS DO DESENVOLVIMENTO NEURO MOTOR',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS DESORDENS DO DESENVOLVIMENTO NEURO MOTOR - Pediátrico',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE C/ COMPROMETIMENTO COGNITIVO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE NO PRÉ/PÓS-OPERATÓRIO DE NEUROCIRURGIA',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE MÉDIO QUEIMADO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM SEQÜELAS POR QUEIMADURAS (MÉDIO E GRANDE QUEIMADOS)'
                )
                THEN 'FISIOTERAPIA ATENDIMENTO (SESSÕES)'
    
                WHEN procedimento IN (
                    'CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA ATENÇÃO ESPECIALIZADA (EXCETO MÉDICO) Adulto - Fisioterapia',
                    'CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA ATENÇÃO ESPECIALIZADA (EXCETO MÉDICO) Pediatrico - Fisioterapia'
                )
                THEN 'FISIOTERAPIA CONSULTA'
    
                WHEN procedimento IN (
                    'ARMAÇÃO PARA ÓCULOS INFANTIL',
                    'ARMAÇÃO PARA ÓCULOS ADULTO',
                    'LENTE COMUM (PAR), GRADUAÇÃO INDETERMINADA. ESFERICA:+4.00/-4.00 CIL: 0.00 A -2.00',
                    'LENTE ESPECIAL (PAR), GRADUAÇÃO INDETERMINADA. ESF: +6.00/-8.00 CIL: 0.00A -3.00',
                    'LENTE BIFOCAL BAIXO (PAR).ESF:-4.00/+7.00 CILÍNDRICO: ATÉ 4.00',
                    'LENTE BIFOCAL MÉDIO (PAR).ESF:-4.00/+7.00 CILÍNDRICO: ATÉ 4.00',
                    'LENTE MULTIFOCAL BAIXO (PAR). ESFERICA: -4.00/+7.00 CILÍNDRICO: ATÉ 4.00',
                    'LENTE MULTIFOCAL MÉDIO (PAR). ESFERICA: -6.00/+7.00 CIL: ATÉ 4.00',
                    'LENTE MULTIFOCAL ALTO (PAR). ESFERICA: -10.00/+6.00 CIL: ATÉ -4.00',
                    'LENTE ESPECIAL ALTO ÍNDICE (PAR). ESFERICA: +/-20.00 CIL: ATÉ -8.00',
                    'LENTE COMUM (PAR), GRADUAÇÃO INDETERMINADA. ESFERICA:+4.00/-4.00 CIL: 0.00 A -2.00 COM LENTE FOTOSSENSÍVEL',
                    'LENTE ESPECIAL (PAR), GRADUAÇÃO INDETERMINADA. ESF: +6.00/- 8.00 CIL: 0.00A -3.00 COM LENTE E FOTOSSENSÍVEL.',
                    'LENTE BIFOCAL BAIXO (PAR). ESF: -4.00/+7.00 CILÍNDRICO: ATÉ 4.00 COM LENTE FOTOSSENSÍVEL',
                    'LENTE BIFOCAL ALTO (PAR). ESF: -10.00/+7.00 CILÍNDRICO: ATÉ - 4.00'
                )
                THEN 'FORNECIMENTO DE ÓCULOS DE GRAU, INCLUINDO A ARMAÇÃO E AS LENTES CORRETIVAS CONFORME INDICAÇÃO MEDICA'
    
                WHEN procedimento IN (
                    'CONSULTA - GINECOLOGIA E COLETA MATERIAL EXAME CITOPATOLOGICO COLO UTERINO C/ RESULTADO E CONSULTA DE RETORNO GINECOLOGIA'
                )
                THEN 'GINECOLOGIA E COLETA MATERIAL EXAME CITOPATOLOGICO COLO UTERINO C/ RESULTADO E CONSULTA DE RETORNO GINECOLOGIA'
    
                WHEN procedimento IN (
                    'MONITORAMENTO PELO SISTEMA HOLTER 24 HS (3 CANAIS)'
                )
                THEN 'HOLTER 24H(3 CANAIS)'
    
                WHEN procedimento IN (
                    'MONITORIZACAO AMBULATORIAL DE PRESSAO ARTERIAL (M.A.P.A)'
                )
                THEN 'M.A.P.A.'
    
                WHEN procedimento IN (
                    'MAMOGRAFIA BILATERAL PARA RASTREAMENTO - Registro no Sistema SISMAMA',
                    'MAMOGRAFIA'
                )
                THEN 'MAMOGRAFIA'
                
                WHEN procedimento IN (
                    'POLISSONOGRAFIA'
                )
                THEN 'POLISSONOGRAFIA'
    
                WHEN procedimento IN (
                    'POTENCIAL EVOCADO AUDITIVO PARA TRIAGEM AUDITIVA (TESTE DA ORELHINHA)'
                )
                THEN 'POTENCIAL EVOCADO AUDITIVO PARA TRIAGEM AUDITIVA (TESTE DA ORELHINHA)'
    
                WHEN procedimento IN (
                    'CERATOMETRIA',
                    'BIOMETRIA ULTRASSÔNICA (MONOCULAR)',
                    'MAPEAMENTO DE RETINA',
                    'MICROSCOPIA ESPECULAR DE CORNEA AMBOS OS OLHOS',
                    'POTENCIAL DE ACUIDADE VISUAL',
                    'RETINOGRAFIA COLORIDA BINOCULAR',
                    'RETINOGRAFIA FLUORESCENTE BINOCULAR',
                    'CAMPIMETRIA COMPUTADORIZADA OU MANUAL COM GRÁFICO',
                    'PAQUIMETRIA ULTRASSÔNICA AMBOS OS OLHOS',
                    'TOPOGRAFIA COMPUTADORIZADA DE CÓRNEA/ CERATOSCOPIA'
                )
                THEN 'PROC. OFTALMOLOGICO'
    
                WHEN procedimento IN (
                    'PRÓTESE PARCIAL MANDIBULAR REMOVÍVEL',
                    'PRÓTESE PARCIAL MAXILAR REMOVÍVEL',
                    'PRÓTESE TOTAL MANDIBULAR',
                    'PRÓTESE TOTAL MAXILAR',
                    'PRÓTESE CORONÁRIAS/INTRARRADICULARES FIXAS/ ADESIVAS (POR ELEMENTO)'
                )
                THEN 'PROTESE DENTARIA'
    
                WHEN procedimento IN (
                    'RADIOGRAFIA BILATERAL DE ORBITAS (PA + OBLIQUAS + HIRTZ)',
                    'RADIOGRAFIA DE ARCADA ZIGOMATICO-MALAR (AP+ OBLIQUAS)',
                    'RADIOGRAFIA DE ARTICULACAO TEMPORO-MANDIBULAR BILATERAL',
                    'RADIOGRAFIA DE CAVUM (LATERAL + HIRTZ)',
                    'RADIOGRAFIA DE CRANIO (PA + LATERAL + OBLÍQUA / BRETTON + HIRTZ)',
                    'RADIOGRAFIA DE CRANIO (PA + LATERAL)',
                    'RADIOGRAFIA DE LARINGE',
                    'RADIOGRAFIA DE MASTOIDE / ROCHEDOS (BILATERAL)',
                    'RADIOGRAFIA DE MAXILAR (PA + OBLIQUA)',
                    'RADIOGRAFIA DE OSSOS DA FACE (MN + LATERAL + HIRTZ)',
                    'RADIOGRAFIA DE REGIAO ORBITARIA (LOCALIZACAO DE CORPO ESTRANHO)',
                    'RADIOGRAFIA DE SEIOS DA FACE (FN + MN + LATERAL + HIRTZ)',
                    'RADIOGRAFIA DE SELA TURSICA (PA + LATERAL + BRETTON)',
                    'RADIOGRAFIA DE COLUNA CERVICAL (AP + LATERAL + TO + OBLIQUAS)',
                    'RADIOGRAFIA DE COLUNA CERVICAL (AP + LATERAL + TO / FLEXAO)',
                    'RADIOGRAFIA DE COLUNA CERVICAL FUNCIONAL / DINAMICA',
                    'RADIOGRAFIA DE COLUNA LOMBO-SACRA',
                    'RADIOGRAFIA DE COLUNA LOMBO-SACRA (C/ OBLIQUAS)',
                    'RADIOGRAFIA DE COLUNA LOMBO-SACRA FUNCIONAL / DINAMICA',
                    'RADIOGRAFIA DE COLUNA TORACICA (AP + LATERAL)',
                    'RADIOGRAFIA DE COLUNA TORACO-LOMBAR',
                    'RADIOGRAFIA DE COLUNA TORACO-LOMBAR DINAMICA',
                    'RADIOGRAFIA DE REGIAO SACRO-COCCIGEA',
                    'RADIOGRAFIA DE CORACAO E VASOS DA BASE (PA + LATERAL + OBLIQUA)',
                    'RADIOGRAFIA DE CORACAO E VASOS DA BASE (PA + LATERAL)',
                    'RADIOGRAFIA DE COSTELAS (POR HEMITORAX)',
                    'RADIOGRAFIA DE ESOFAGO',
                    'RADIOGRAFIA DE ESTERNO',
                    'RADIOGRAFIA DE MEDIASTINO (PA E PERFIL)',
                    'RADIOGRAFIA DE PNEUMOMEDIASTINO',
                    'RADIOGRAFIA DE TORAX (APICO-LORDORTICA)',
                    'RADIOGRAFIA DE TÓRAX (PA + INSPIRAÇÃO + EXPIRAÇÃO + LATERAL)',
                    'RADIOGRAFIA DE TORAX (PA + LATERAL + OBLIQUA)',
                    'RADIOGRAFIA DE TORAX (PA + PERFIL)',
                    'RADIOGRAFIA DE TORAX (PA PADRAO OIT)',
                    'RADIOGRAFIA DE TORAX (PA)',
                    'RADIOGRAFIA DE ANTEBRACO',
                    'RADIOGRAFIA DE ARTICULACAO ACROMIO-CLAVICULAR',
                    'RADIOGRAFIA DE ARTICULACAO ESCAPULO-UMERAL',
                    'RADIOGRAFIA DE ARTICULACAO ESTERNO-CLAVICULAR',
                    'RADIOGRAFIA DE BRACO',
                    'RADIOGRAFIA DE CLAVICULA',
                    'RADIOGRAFIA DE COTOVELO',
                    'RADIOGRAFIA DE DEDOS DA MAO',
                    'RADIOGRAFIA DE MAO',
                    'RADIOGRAFIA DE MAO E PUNHO (P/ DETERMINACAO DE IDADE OSSEA)',
                    'RADIOGRAFIA DE ESCAPULA/OMBRO (TRES POSICOES)',
                    'RADIOGRAFIA DE PUNHO (AP + LATERAL + OBLIQUA)',
                    'RADIOGRAFIA DE ABDOMEN (AP + LATERAL / LOCALIZADA)',
                    'RADIOGRAFIA DE ABDOMEN AGUDO (MINIMO DE 3 INCIDENCIAS)',
                    'RADIOGRAFIA DE ABDOMEN SIMPLES (AP)',
                    'RADIOGRAFIA DE ESTOMAGO E DUODENO',
                    'RADIOGRAFIA DE INTESTINO DELGADO (TRANSITO)',
                    'RADIOGRAFIA DE ARTICULACAO COXO-FEMORAL',
                    'RADIOGRAFIA DE ARTICULACAO SACRO-ILIACA',
                    'RADIOGRAFIA DE ARTICULACAO TIBIO-TARSICA',
                    'RADIOGRAFIA DE BACIA',
                    'RADIOGRAFIA DE CALCANEO',
                    'RADIOGRAFIA DE JOELHO (AP + LATERAL)',
                    'RADIOGRAFIA DE JOELHO OU PATELA (AP + LATERAL + AXIAL)',
                    'RADIOGRAFIA DE JOELHO OU PATELA (AP + LATERAL + OBLIQUA + 3 AXIAIS)',
                    'RADIOGRAFIA DE PE / DEDOS DO PE',
                    'RADIOGRAFIA DE PERNA',
                    'BRONCOGRAFIA UNILATERAL',
                    'DUCTOGRAFIA (POR MAMA)',
                    'COLANGIOGRAFIA PER-OPERATORIA',
                    'COLANGIOGRAFIA POS-OPERATORIA',
                    'HISTEROSSALPINGOGRAFIA'
                )
                THEN 'RADIOGRAFIA'
    
                WHEN procedimento IN (
                    'RESSONANCIA MAGNETICA DE ARTICULACAO TEMPORO-MANDIBULAR (BILATERAL)',
                    'RESSONANCIA MAGNETICA DE COLUNA CERVICAL/PESCOÇO',
                    'RESSONANCIA MAGNETICA DE COLUNA LOMBO-SACRA',
                    'RESSONANCIA MAGNETICA DE COLUNA TORACICA',
                    'RESSONÂNCIA MAGNÉTICA DA MAMA BILATERAL',
                    'RESSONANCIA MAGNETICA DE CRANIO',
                    'RESSONANCIA MAGNETICA DE SELA TURCICA',
                    'RESSONANCIA MAGNETICA CARDÍACA – MORFOLÓGICA E FUNCIONAL',
                    'RESSONANCIA MAGNETICA CARDÍACA – MORFOLÓGICA E FUNCIONAL E PERFUSÃO',
                    'RESSONANCIA MAGNETICA DE MEMBRO SUPERIOR (UNILATERAL)',
                    'RESSONANCIA MAGNETICA DE TORAX',
                    'RESSONANCIA MAGNETICA DE ABDOMEN SUPERIOR',
                    'RESSONANCIA MAGNETICA DE BACIA / PELVE / ABDOMEN INFERIOR',
                    'RESSONANCIA DE PROSTATA',
                    'RESSONANCIA MAGNETICA DE MEMBRO INFERIOR (UNILATERAL)',
                    'RESSONANCIA MAGNETICA DE VIAS BILIARES/COLANGIORRESSONANCIA',
                    'ANGIORESSONANCIA CEREBRAL'
                )
                THEN 'RESSONANCIA MAGNETICA'
                
                WHEN procedimento IN (
                    'TESTE DE ESFORCO / TESTE ERGOMETRICO'
                )
                THEN 'TESTE DE ESFORCO / TESTE ERGOMETRICO'
    
                WHEN procedimento IN (
                    'TOMOGRAFIA COMPUTADORIZADA DE COLUNA CERVICAL',
                    'TOMOGRAFIA COMPUTADORIZADA DE COLUNA LOMBO-SACRA',
                    'TOMOGRAFIA COMPUTADORIZADA DE COLUNA TORACICA',
                    'TOMOGRAFIA COMPUTADORIZADA DE FACE / SEIOS DA FACE / ARTICULACOES TEMPORO-MANDIBULARES',
                    'TOMOGRAFIA COMPUTADORIZADA DO PESCOCO',
                    'TOMOGRAFIA COMPUTADORIZADA DE SELA TURCICA',
                    'TOMOGRAFIA COMPUTADORIZADA DO CRANIO',
                    'TOMOGRAFIA COMPUTADORIZADA DE CRANIO',
                    'TOMOGRAFIA COMPUTADORIZADA DE SEGMENTOS APENDICULARES - (BRACO, ANTEBRAÇO, MÃO, COXA, PERNA, PÉ)',
                    'TOMOGRAFIA COMPUTADORIZADA DE TORAX',
                    'TOMOGRAFIA COMPUTADORIZADA DE ABDOMEN SUPERIOR',
                    'TOMOGRAFIA COMPUTADORIZADA DE ARTICULACOES DE MEMBRO INFERIOR',
                    'TOMOGRAFIA COMPUTADORIZADA DE PELVE / BACIA / ABDOMEN INFERIOR',
                    'TOMOGRAFIA DE COERÊNCIA ÓPTICA',
                    'TOMOGRAFIA COMPUTADORIZADA DE ARTICULACOES DE MEMBRO SUPERIOR'
                )
                THEN 'TOMOGRAFIA'
    
                WHEN procedimento IN (
                    'ULTRASSONOGRAFIA DE ABDÔMEN SUPERIOR',
                    'ULTRASSONOGRAFIA DE ABDOMEN TOTAL',
                    'ULTRASSONOGRAFIA DE APARELHO URINÁRIO',
                    'ULTRASSONOGRAFIA DE PARTES MOLES',
                    'ULTRASSONOGRAFIA DE ARTICULACAO',
                    'ULTRASSONOGRAFIA DE BOLSA ESCROTAL',
                    'ULTRASSONOGRAFIA DE GLOBO OCULAR / ORBITA (MONOCULAR)',
                    'ULTRASSONOGRAFIA MAMARIA BILATERAL',
                    'ULTRASSONOGRAFIA DE ARTICULACAO E PARTES MOLES',
                    'ULTRASSONOGRAFIA DE PROSTATA POR VIA ABDOMINAL COM MEDIDA DE RESIDUO POS MICCIONAL',
                    'ULTRASSONOGRAFIA DE PROSTATA (VIA TRANSRETAL)',
                    'ULTRASSONOGRAFIA DE TIREOIDE',
                    'ULTRASSONOGRAFIA OBSTETRICA',
                    'ULTRASSONOGRAFIA OBSTÉTRICA MORFOLOGICA POR FETO',
                    'ULTRASSONOGRAFIA PELVICA (GINECOLOGICA)',
                    'ULTRASSONOGRAFIA TRANSFONTANELA',
                    'ULTRASSONOGRAFIA TRANSVAGINAL',
                    'ECOCARDIOGRAFIA (FETAL)',
                    'ULTRASSONOGRAFIA DE PROSTATA POR VIA ABDOMINAL'
                )
                THEN 'ULTRASSONOGRAFIA'
    
                WHEN procedimento IN (
                    'ECOCARDIOGRAFIA DOPPLER COLORIDO',
                    'ULTRASSONOGRAFIA BOLSA ESCROTAL COM DOPPLER',
                    'ULTRASSONOGRAFIA DOPPLER DE ARTERIAS POPLITEA',
                    'ULTRASSONOGRAFIA DOPPLER DE ARTERIAS RENAIS',
                    'ULTRASSONOGRAFIA DOPPLER DE PORTA HEPATICA',
                    'ULTRASSONOGRAFIA DOPPLER DE VEIAS ILIACAS',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO INFERIOR ARTERIAL',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO INFERIOR VENOSO',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO SUPERIOR ARTERIAL',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO SUPERIOR VENOSO',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO TRANSVAGINAL',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE PENIS',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE TIREOIDE',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - CAROTIDAS E VERTEBRAIS',
                    'ULTRASSONOGRAFIA DOPPLER DE FLUXO OBSTETRICO',
                    'ULTRASSONOGRAFIA OBSTETRICA C/ DOPPLER COLORIDO E PULSADO'
                )
                THEN 'US DOPPLER'
    
                WHEN procedimento IN (
                    'TESTES VESTIBULARES / OTONEUROLOGICOS'
                )
                THEN 'VECTOELETRONISTAGMOGRAFIA/ testes vestibulares'
    
                WHEN procedimento IN (
                    'HISTEROSCOPIA (VIDEO HISTEROSCOPIA DIAGNÓSTICA) (SEM  SEDAÇÃO)',
                    'HISTEROSCOPIA (VIDEO HISTEROSCOPIA DIAGNOSTICA) COM BIOPSIA E RESULTADO DA BIÓPSIA (SEM  SEDAÇÃO)'
                )
                THEN 'VIDEOHISTEROSCOPIA'
    
                WHEN procedimento IN (
                    'VIDEOLARINGOSCOPIA'
                )
                THEN 'VIDEOLARINGOSCOPIA'
    
                ELSE 'outro'
            END AS grupo,
    
            cidade
    
        FROM agendamentos
    
        WHERE situacao = 'VALIDADO'
    )
    SELECT
        grupo AS procedimento,
        SUM(CASE WHEN cidade = 'Itaguaí' THEN 1 ELSE 0 END) AS IT,
        SUM(CASE WHEN cidade = 'Japeri' THEN 1 ELSE 0 END) AS JAP,
        SUM(CASE WHEN cidade = 'Mesquita' THEN 1 ELSE 0 END) AS MESQ,
        SUM(CASE WHEN cidade = 'Nilópolis' THEN 1 ELSE 0 END) AS NILOPOLIS,
        SUM(CASE WHEN cidade = 'Nova Iguaçu' THEN 1 ELSE 0 END) AS NI,
        SUM(CASE WHEN cidade = 'Paracambi' THEN 1 ELSE 0 END) AS PARA,
        SUM(CASE WHEN cidade = 'Queimados' THEN 1 ELSE 0 END) AS QUEI,
        COUNT(*) AS TOTAL
    
    FROM dados
    
    GROUP BY grupo
    
    ORDER BY grupo;
""").show()

+--------------------+---+---+----+---------+---+----+----+-----+
|        procedimento| IT|JAP|MESQ|NILOPOLIS| NI|PARA|QUEI|TOTAL|
+--------------------+---+---+----+---------+---+----+----+-----+
|AUDIOMETRIA TONAL...|112|  0|   0|        2|  0|   0|   1|  115|
|        BIÓPSIA MAMA|  0|  0| 120|       19|  0|   0|   0|  139|
|CIRURGIA OFTALMOL...|152|  0|   3|        0|  0|  48|  44|  247|
|CONSULTAS CARDIO ...|  0|  0|   0|       26|  0|   0|   0|   26|
|CONSULTAS ESPECIA...|  1|  0|   0|       17| 78|   0|   2|   98|
|CONSULTAS OFTALMO...|737|  0|   0|        0|  0| 658|  75| 1470|
|       DESINTOMETRIA| 57|  1|   0|        1|  0|   0|   2|   61|
|ECOCARDIOGRAFIA T...|  0|  0|   0|       19|472|   0|   7|  498|
| ELETROENCEFALOGRAMA|  0|  1|   0|       12| 38|   0|   6|   57|
|ELETRONEUROMIOGRAFIA| 24|  0|   0|        6| 51|   0|   6|   87|
|ENDOSCOPIAS DIGES...|145| 37| 291|       95|119|   1|  20|  708|
|        ESPIROMETRIA|  0|  1|  32|        0|  0|   0|   2|   35|
|FISIOTERA

# PROCEDIMENTOS NAO LISTADOS

In [14]:
spark.sql("""
    WITH dados AS (
        SELECT
            CASE
                WHEN procedimento IN (
                    'LOGOAUDIOMETRIA (LDV-IRF-LRF)',
                    'AUDIOMETRIA TONAL LIMIAR (VIA AEREA / OSSEA) IMPEDANCIOMETRIA E LOGOAUDIOMETRIA'
                )
                THEN 'AUDIOMETRIA TONAL LIMIAR IMPED e LOGOAUDOMETRIA (LDV-IRF-LRF)'
    
                WHEN procedimento IN (
                    'CORE BIOPSIA MAMA POR ULTRASSONOGRAFIA (INCLUI RESULTADO DO HISTOPATOLOGICO - 4 FRAGMENTOS)'
                )
                THEN 'BIÓPSIA MAMA'
    
                WHEN procedimento IN (
                    'BIOPSIA PROSTATA TRANSRETAL GUIADA POR ULTRASSONOGRAFIA- (ATE 8 FRAGMENTOS) (INCLUI RESULTADO DO HISTOPATOLOGICO)'
                )
                THEN 'BIÓPSIA PRÓSTATA'
                
                WHEN procedimento IN (
                    'FACOEMULSIFICACAO C/ IMPLANTE DE LENTE INTRA-OCULAR DOBRAVEL',
                    'CAPSULOTOMIA A YAG LASER - MONOCULAR',
                    'TRATAMENTO CIRURGICO DE PTERIGIO',
                    'EXERESE DE CALAZIO E OUTRAS PEQUENAS LESOES DA PALPEBRA E SUPERCILIOS'
                )
                THEN 'CIRURGIA OFTALMOLOGICA'
    
                WHEN procedimento IN (
                    'CONSULTA -  OFTALMOLOGISTA COM TONOMETRIA E FUNDOSCOPIA',
                    'CONSULTA - OFTALMOLOGISTA'
                )
                THEN 'CONSULTAS OFTALMOLOGIA'
                
                WHEN procedimento IN (
                    'COLPOSCOPIA COM BIOPSIA E RESULTADO DA BIOPSIA',
                    'COLPOSCOPIA'
                )
                THEN 'COLPOSCOPIA COM BIOPSIA E RESULTADO DA BIOPSIA'
    
                WHEN procedimento IN (
                    'CONSULTA - CARDIOLOGIA COM ELETROCARDIOGRAMA'
                )
                THEN 'CONSULTAS CARDIO C/ ELETRO'
    
                WHEN procedimento IN (
                    'CONSULTA - UROLOGISTA',
                    'CONSULTA - REUMATOLOGISTA',
                    'CONSULTA - PSIQUIATRA - PEDIATRICO',
                    'CONSULTA - PSIQUIATRA',
                    'CONSULTA - PROCTOLOGISTA',
                    'CONSULTA - PNEUMOLOGISTA',
                    'CONSULTA - PNEUMOLOGISTA -PEDIATRICO',
                    'CONSULTA - ORTOPEDISTA E TRAUMATOLOGISTA',
                    'CONSULTA - OTORRINOLARINGOLOGISTA',
                    'CONSULTA - DERMATOLOGISTA',
                    'CONSULTA - ENDOCRINOLOGISTA e METABOLISTA',
                    'CONSULTA - ENDOCRINOLOGISTA e METABOLISTA - PEDIATRICO',
                    'CONSULTA - GASTROENTEROLOGISTA',
                    'CONSULTA - GASTROENTEROLOGISTA - PEDIATRICO',
                    'CONSULTA - GERIATRA',
                    'CONSULTA - GINECOLOGIA',
                    'CONSULTA - OBSTÉTRICA',
                    'CONSULTA - HEMATOLOGISTA',
                    'CONSULTA - HEMATOLOGISTA - PEDIATRICO',
                    'CONSULTA - INFECTOLOGISTA',
                    'CONSULTA - MASTOLOGISTA',
                    'CONSULTA - NEFROLOGISTA',
                    'CONSULTA - NEUROCIRURGIÃO',
                    'CONSULTA - NEUROLOGISTA',
                    'CONSULTA - NEUROLOGISTA PEDIATRICO',
                    'CONSULTA - CIRURGIAO GERAL',
                    'CONSULTA - PEDIATRIA',
                    'CONSULTA - CLINICO',
                    'CONSULTA - CARDIOLOGISTA',
                    'CONSULTA - CARDIOLOGISTA - PEDIATRICO',
                    'CONSULTA - ANGIOLOGISTA',
                    'CONSULTA - ALERGISTA e IMUNOLOGISTA'
                )
                THEN 'CONSULTAS ESPECIALIZADAS'
    
                WHEN procedimento IN (
                    'DENSITOMETRIA OSSEA DUO-ENERGETICA DE COLUNA (VERTEBRAS LOMBARES E/OU FEMUR)'
                )
                THEN 'DESINTOMETRIA'
    
                WHEN procedimento IN (
                    'ECOCARDIOGRAFIA TRANSTORACICA - PEDIATRICO (0 A 13 ANOS)',
                    'ECOCARDIOGRAFIA TRANSTORACICA'
                )
                THEN 'ECOCARDIOGRAFIA TRANSTORACICA'
    
                WHEN procedimento IN (
                    'ELETROENCEFALOGRAMA EM SONO INDUZIDO C/ OU S/ MEDICAMENTO (EEG)',
                    'ELETROENCEFALOGRAMA EM VIGILIA E SONO ESPONTANEO C/ OU S/ FOTOESTIMULO (EEG)',
                    'ELETROENCEFALOGRAMA QUANTITATIVO C/ MAPEAMENTO (EEG)'
                )
                THEN 'ELETROENCEFALOGRAMA'
    
                WHEN procedimento IN (
                    'ELETRONEUROMIOGRAMA (ENMG) CADA MEMBRO'
                )
                THEN 'ELETRONEUROMIOGRAFIA'
    
                WHEN procedimento IN (
                    'COLONOSCOPIA (COLOSCOPIA) SEM BIOPSIA',
                    'COLONOSCOPIA COM BIOPSIA E POLIPECTOMIA - INCLUINDO O RESULTADO DO HISTOPATOLOGICO',
                    'COLONOSCOPIA COM BIOPSIA - INCLUINDO RESULTADO DO HISTOPATOLOGICO',
                    'ESOFAGOGASTRODUODENOSCOPIA COM BIÓPSIAS INCLUINDO RESULTADO DO HISTOPATOLÓGICO',
                    'ESOFAGOGASTRODUODENOSCOPIA - SEM BIÓPSIA',
                    'ESOFAGOGASTRODUODENOSCOPIA COM BIÓPSIAS E POLIPECTOMIA INCLUINDO RESULTADO DO HISTOPATOLÓGICO'
                )
                THEN 'ENDOSCOPIAS DIGESTIVAS (ALTA E BAIXA)'
    
                WHEN procedimento IN (
                    'ESPIROMETRIA OU PROVA DE FUNCAO PULMONAR COMPLETA COM BRONCODILATADOR'
                )
                THEN 'ESPIROMETRIA'
    
                WHEN procedimento IN (
                    'DOSAGEM DE ACIDO URICO',
                    'DOSAGEM DE ACIDO VANILMANDELICO',
                    'DOSAGEM DE ALDOLASE',
                    'DOSAGEM DE AMILASE',
                    'DOSAGEM DE AMONIA',
                    'DOSAGEM DE BILIRRUBINA TOTAL E FRACOES',
                    'DOSAGEM DE CALCIO',
                    'DOSAGEM DE CALCIO IONIZAVEL',
                    'DOSAGEM DE CLORETO',
                    'DOSAGEM DE COLESTEROL HDL',
                    'DOSAGEM DE COLESTEROL LDL',
                    'DOSAGEM DE COLESTEROL TOTAL',
                    'DOSAGEM DE COLINESTERASE',
                    'DOSAGEM DE CREATININA',
                    'DOSAGEM DE CREATINOFOSFOQUINASE (CPK)',
                    'DOSAGEM DE CREATINOFOSFOQUINASE FRACAO MB',
                    'DOSAGEM DE DESIDROGENASE LATICA',
                    'DOSAGEM DE FERRITINA',
                    'DOSAGEM DE FERRO SERICO',
                    'DOSAGEM DE FOSFATASE ALCALINA',
                    'DOSAGEM DE GALACTOSE',
                    'DOSAGEM DE GAMA-GLUTAMIL-TRANSFERASE (GAMA GT)',
                    'DOSAGEM DE GLICOSE',
                    'DOSAGEM DE GLICOSE-6-FOSFATO DESIDROGENASE',
                    'DOSAGEM DE HEMOGLOBINA GLICOSILADA',
                    'DOSAGEM DE LIPASE',
                    'DOSAGEM DE MAGNESIO',
                    'DOSAGEM DE POTASSIO',
                    'DOSAGEM DE PROTEINAS TOTAIS',
                    'DOSAGEM DE PROTEINAS TOTAIS E FRACOES',
                    'DOSAGEM DE SODIO',
                    'DOSAGEM DE TRANSAMINASE GLUTAMICO-OXALACETICA (TGO)',
                    'DOSAGEM DE TRANSAMINASE GLUTAMICO-PIRUVICA (TGP)',
                    'DOSAGEM DE TRANSFERRINA',
                    'DOSAGEM DE TRIGLICERIDEOS',
                    'DOSAGEM DE UREIA',
                    'DOSAGEM DE VITAMINA B12',
                    'DOSAGEM DE 25 HIDROXIVITAMINA D',
                    'DOSAGEM DE FATOR IX',
                    'DOSAGEM DE FATOR V',
                    'DOSAGEM DE FATOR VII',
                    'DOSAGEM DE FATOR VIII',
                    'DOSAGEM DE FATOR VIII (INIBIDOR)',
                    'DOSAGEM DE FATOR VON WILLEBRAND (ANTIGENO)',
                    'DOSAGEM DE FATOR X',
                    'DOSAGEM DE FATOR XI',
                    'DOSAGEM DE FATOR XII',
                    'DOSAGEM DE FATOR XIII',
                    'DOSAGEM DE FIBRINOGENIO',
                    'DOSAGEM DE HEMOGLOBINA',
                    'DOSAGEM DE HEMOGLOBINA FETAL',
                    'DOSAGEM DE PROTEINA C FUNCIONAL',
                    'DOSAGEM DE PROTEINA S FUNCIONAL',
                    'DOSAGEM DE ALFA-FETOPROTEINA',
                    'DOSAGEM DE ANTIGENO PROSTATICO ESPECIFICO (PSA)',
                    'DOSAGEM DE COMPLEMENTO C3',
                    'DOSAGEM DE COMPLEMENTO C4',
                    'DOSAGEM DE IMUNOGLOBULINA A (IGA)',
                    'DOSAGEM DE IMUNOGLOBULINA E (IGE)',
                    'DOSAGEM DE IMUNOGLOBULINA M (IGM)',
                    'DOSAGEM DE PROTEINA C REATIVA',
                    'DOSAGEM DA FRAÇÃO C1Q DO COMPLEMENTO',
                    'DOSAGEM DO ANTÍGENO CA 125',
                    'DOSAGEM DE ANTI-BETA-2-GLICOPROTEÍNA I - IGM',
                    'DOSAGEM DE PROTEINAS (URINA DE 24 HORAS)',
                    'DOSAGEM DE 17-ALFA-HIDROXIPROGESTERONA',
                    'DOSAGEM DE 17-HIDROXICORTICOSTEROIDES',
                    'DOSAGEM DE ACIDO 5-HIDROXI-INDOL-ACETICO (SEROTONINA)',
                    'DOSAGEM DE ADRENOCORTICOTROFICO (ACTH)',
                    'DOSAGEM DE ANDROSTENEDIONA',
                    'DOSAGEM DE CALCITONINA',
                    'DOSAGEM DE CORTISOL',
                    'DOSAGEM DE DEHIDROEPIANDROSTERONA (DHEA)',
                    'DOSAGEM DE DIHIDROTESTOTERONA (DHT)',
                    'DOSAGEM DE ESTRADIOL',
                    'DOSAGEM DE ESTRIOL',
                    'DOSAGEM DE ESTRONA',
                    'DOSAGEM DE GASTRINA',
                    'DOSAGEM DE GLOBULINA TRANSPORTADORA DE TIROXINA',
                    'DOSAGEM DE GONADOTROFINA CORIONICA HUMANA (HCG, BETA HCG)',
                    'DOSAGEM DE HORMONIO DE CRESCIMENTO (HGH)',
                    'DOSAGEM DE HORMONIO FOLICULO-ESTIMULANTE (FSH)',
                    'DOSAGEM DE HORMONIO LUTEINIZANTE (LH)',
                    'DOSAGEM DE HORMONIO TIREOESTIMULANTE (TSH)',
                    'DOSAGEM DE INSULINA',
                    'DOSAGEM DE PARATORMONIO',
                    'DOSAGEM DE PROGESTERONA',
                    'DOSAGEM DE PROLACTINA',
                    'DOSAGEM DE SOMATOMEDINA C (IGF1)',
                    'DOSAGEM DE SULFATO DE HIDROEPIANDROSTERONA (DHEAS)',
                    'DOSAGEM DE TESTOSTERONA',
                    'DOSAGEM DE TESTOSTERONA LIVRE',
                    'DOSAGEM DE TIREOGLOBULINA',
                    'DOSAGEM DE TIROXINA (T4)',
                    'DOSAGEM DE TIROXINA LIVRE (T4 LIVRE)',
                    'DOSAGEM DE TRIIODOTIRONINA (T3)',
                    'DOSAGEM DE ACIDO MANDELICO',
                    'DOSAGEM DE ACIDO VALPROICO',
                    'DOSAGEM DE BARBITURATOS',
                    'DOSAGEM DE BENZODIAZEPINICOS',
                    'DOSAGEM DE CADMIO',
                    'DOSAGEM DE CARBAMAZEPINA',
                    'DOSAGEM DE COBRE',
                    'DOSAGEM DE FENITOINA',
                    'DOSAGEM DE LITIO',
                    'DOSAGEM DE ZINCO',
                    'DOSAGEM DE FRUTOSE',
                    'DOSAGEM DE FENILALANINA (CONTROLE / DIAGNOSTICO TARDIO)',
                    'DOSAGEM DE 17 HIDROXI PROGESTERONA EM PAPEL DE FILTRO (COMPONENTE DO TESTE DO PEZINHO)',
                    'DETERMINAÇÃO DE TEMPO DE COAGULAÇÃO',
                    'DETERMINAÇÃO DE TEMPO DE SANGRAMENTO -DUKE',
                    'DETERMINAÇÃO DE TEMPO DE SANGRAMENTO DE IVY',
                    'DETERMINAÇÃO DE TEMPO DE TROMBINA',
                    'DETERMINAÇÃO DE TEMPO DE TROMBOPLASTINA PARCIAL ATIVADA (TTP ATIVADA)',
                    'DETERMINAÇÃO DE TEMPO E ATIVIDADE DA PROTROMBINA (TAP)',
                    'DETERMINAÇÃO DE VELOCIDADE DE HEMOSSEDIMENTAÇÃO (VHS)',
                    'DETERMINAÇÃO DE COMPLEMENTO (CH50)',
                    'DETERMINAÇÃO DE FATOR REUMATOIDE',
                    'DETERMINAÇÃO QUANTITATIVA DE PROTEÍNA C REATIVA',
                    'DETERMINAÇÃO DE ÍNDICE DE TIROXINA LIVRE',
                    'DETERMINAÇÃO DE T3 REVERSO',
                    'PESQUISA DE CELULAS LE',
                    'PESQUISA DE CORPUSCULOS DE HEINZ',
                    'PESQUISA DE HEMOGLOBINA S',
                    'PESQUISA DE ANTICOAGULANTE LÚPICO',
                    'PESQUISA DE ANTICORPOS ANTI-HIV-1 (WESTERN BLOT)',
                    'PESQUISA DE ANTICORPOS ANTI-HIV-1 + HIV-2 (ELISA)',
                    'PESQUISA DE ANTICORPOS ANTI-HTLV-1 + HTLV-2',
                    'PESQUISA DE ANTICORPOS ANTI-RIBONUCLEOPROTEINA (RNP)',
                    'PESQUISA DE ANTICORPOS ANTI-SS-A (RO)',
                    'PESQUISA DE ANTICORPOS ANTI-SS-B (LA)',
                    'PESQUISA DE ANTICORPOS ANTIESTREPTOLISINA O (ASLO)',
                    'PESQUISA DE ANTICORPOS ANTITIREOGLOBULINA',
                    'PESQUISA DE ANTICORPOS CONTRA ANTIGENO DE SUPERFICIE DO VIRUS DA HEPATITE B (ANTI-HBS)',
                    'PESQUISA DE ANTICORPOS CONTRA ANTIGENO E DO VIRUS DA HEPATITE B (ANTI-HBE)',
                    'PESQUISA DE ANTICORPOS CONTRA O VIRUS DA HEPATITE C (ANTI-HCV)',
                    'PESQUISA DE ANTICORPOS CONTRA O VIRUS DO SARAMPO',
                    'PESQUISA DE ANTICORPOS HETEROFILOS CONTA O VIRUS EPSTEIN-BARR',
                    'PESQUISA DE ANTICORPOS IGG ANTICITOMEGALOVIRUS',
                    'PESQUISA DE ANTICORPOS IGG ANTITOXOPLASMA',
                    'PESQUISA DE ANTICORPOS IGG E IGM CONTRA ANTIGENO CENTRAL DO VIRUS DA HEPATITE B (ANTI-HBC-TOTAL)',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS DA HEPATITE A (HAV-IGG)',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS DA RUBEOLA',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS DA VARICELA-HERPES ZOSTER',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS EPSTEIN-BARR',
                    'PESQUISA DE ANTICORPOS IGG CONTRA O VIRUS HERPES SIMPLES',
                    'PESQUISA DE ANTICORPOS IGM CONTRA ANTIGENO CENTRAL DO VIRUS DA HEPATITE B (ANTI-HBC-IGM)',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS DA HEPATITE A (HAV-IGG)',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS DA RUBEOLA',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS DA VARICELA-HERPES ZOSTER',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS EPSTEIN-BARR',
                    'PESQUISA DE ANTICORPOS IGM CONTRA O VIRUS HERPES SIMPLES',
                    'PESQUISA DE ANTIGENO CARCINOEMBRIONARIO (CEA)',
                    'PESQUISA DE ANTIGENO DE SUPERFICIE DO VIRUS DA HEPATITE B (HBSAG)',
                    'PESQUISA DE ANTIGENO E DO VIRUS DA HEPATITE B (HBEAG)',
                    'PESQUISA DE CRIOGLOBULINAS',
                    'PESQUISA DE FATOR REUMATOIDE (WAALER-ROSE)',
                    'PESQUISA DE IMUNOGLOBULINA E (IGE) ALERGENO-ESPECIFICA',
                    'PESQUISA DE ANTICORPOS ANTICORPOS ANTI-HTLV-1 (WESTERN-BLOT)',
                    'PESQUISA DE EOSINOFILOS',
                    'PESQUISA DE GORDURA FECAL',
                    'PESQUISA DE LARVAS NAS FEZES',
                    'PESQUISA DE LEUCOCITOS NAS FEZES',
                    'PESQUISA DE LEVEDURAS NAS FEZES',
                    'PESQUISA DE OVOS E CISTOS DE PARASITAS',
                    'PESQUISA DE SANGUE OCULTO NAS FEZES',
                    'PESQUISA DE TREPONEMA PALLIDUM',
                    'PESQUISA DE ESPERMATOZOIDES (APOS VASECTOMIA)',
                    'PROVA DO LATEX P/ PESQUISA DO FATOR REUMATOIDE',
                    'PESQUISA DE FATOR RH (INCLUI D FRACO)',
                    'DETERMINACAO DE CAPACIDADE DE FIXACAO DO FERRO',
                    'DETERMINACAO DE CURVA GLICEMICA CLASSICA (5 DOSAGENS)',
                    'DETERMINACAO DE ANTICORPOS ANTIPLAQUETARIOS',
                    'DETERMINACAO DIRETA E REVERSA DE GRUPO ABO',
                    'BACILOSCOPIA DIRETA P/ BAAR TUBERCULOSE (DIAGNÓSTICA)',
                    'BACILOSCOPIA DIRETA P/ BAAR (HANSENIASE)',
                    'BACILOSCOPIA DIRETA P/ BAAR TUBERCULOS (CONTROLE)',
                    'TESTE DIRETO DE ANTIGLOBULINA HUMANA (TAD)',
                    'ANTIBETA 2 GLICOPROTEINA I -IGG',
                    'ANTIBIOGRAMA',
                    'ANTIBIOGRAMA P/ MICOBACTERIAS',
                    'TESTE INDIRETO DE ANTIGLOBULINA HUMANA (TIA)',
                    'EXAME ANATOMO-PATOLOGICO DO COLO UTERINO - PECA CIRURGICA',
                    'EXAME ANATOMO-PATOLÓGICO PARA CONGELAMENTO / PARAFINA POR PEÇA CIRURGICA OU POR BIOPSIA (EXCETO COLO UTERINO E MAMA)',
                    'EXAME ANATOMOPATOLOGICO DE MAMA - BIOPSIA',
                    'EXAME ANATOMOPATOLOGICO DE MAMA - PECA CIRURGICA',
                    'EXAME ANATOMO-PATOLOGICO DO COLO UTERINO - BIOPSIA',
                    'CONTAGEM DE PLAQUETAS',
                    'CONTAGEM DE RETICULOCITOS',
                    'CONTAGEM DE LINFOCITOS B',
                    'CONTAGEM DE LINFOCITOS CD4/CD8',
                    'CONTAGEM DE LINFOCITOS T TOTAIS',
                    'TESTE DE TOLERANCIA A INSULINA / HIPOGLICEMIANTES ORAIS',
                    'ELETROFORESE DE HEMOGLOBINA',
                    'ERITROGRAMA (ERITROCITOS, HEMOGLOBINA, HEMATOCRITO)',
                    'HEMATOCRITO',
                    'HEMOGRAMA COMPLETO',
                    'LEUCOGRAMA',
                    'PROVA DE RETRACAO DO COAGULO',
                    'PROVA DO LAÇO',
                    'DETECÇÃO DE ÁCIDOS NUCLEICOS DO HIV-1 (QUALITATIVO)',
                    'DETECÇÃO DE RNA DO VÍRUS DA HEPATITE C (QUALITATIVO)',
                    'DETECÇÃO DE RNA DO HTLV-1',
                    'EXAME COPROLOGICO FUNCIONAL',
                    'ANALISE DE CARACTERES FISICOS, ELEMENTOS E SEDIMENTO DA URINA',
                    'CLEARANCE DE CREATININA',
                    'CLEARANCE DE UREIA',
                    'TESTE P/ INVESTIGACAO DO DIABETES INSIPIDUS',
                    'BACTERIOSCOPIA (GRAM)',
                    'CULTURA DE BACTERIAS P/ IDENTIFICACAO',
                    'CULTURA PARA BAAR',
                    'HEMOCULTURA',
                    'EXAME CITOPATOLOGICO CERVICO-VAGINAL/MICROFLORA- Registro no SISCOLO',
                    'EXAME CITOPATOLÓGICO CERVICO-VAGINAL/MICROFLORA',
                    'IMUNOHISTOQUIMICA DE NEOPLASIAS MALIGNAS (POR MARCADOR)',
                    'ELETROFORESE DE PROTEINAS'
                )
                THEN 'EXAMES LABORATORIAIS'
    
                WHEN procedimento IN (
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE NO PRÉ/PÓS CIRURGIAS UROGINECOLÓGICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES C/ DISFUNÇÕES UROGINECOLÓGICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE ONCOLÓGICO CLÍNICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE NO PRÉ E PÓS CIRURGIA ONCOLÓGICA',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM ALTERAÇÕES OCULOMOTORAS CENTRAIS C/ COMPROMETIMENTO SISTÊMICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM ALTERAÇÕES OCULOMOTORAS CENTRAIS C/ COMPROMETIMENTO SISTÊMICO - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM ALTERAÇÕES OCULOMOTORAS PERIFÉRICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM ALTERAÇÕES OCULOMOTORAS PERIFÉRICAS - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO RESPIRATÓRIO COM COMPLICAÇÕES SISTÊMICAS - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO RESPIRATÓRIO COM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO RESPIRATÓRIO SEM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM TRANSTORNO CLÍNICO CARDIOVASCULAR',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE PRÉ/PÓS CIRURGIA CARDIOVASCULAR',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS DISFUNÇÕES VASCULARES PERIFÉRICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES NO PRÉ E PÓS-OPERATÓRIO NAS DISFUNÇÕES MÚSCULO ESQUELÉTICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS ALTERAÇÕES MOTORAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES NO PRÉ E PÓS-OPERATÓRIO NAS DISFUNÇÕES MUSCULO- ESQUELETICAS C/ COMPLICAÇÕES SISTEMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM DISTÚRBIOS NEURO-CINÉTICO-FUNCIONAIS SEM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM DISTÚRBIOS NEURO-CINÉTICO-FUNCIONAIS COM COMPLICAÇÕES SISTÊMICAS',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTES COM DISTÚRBIOS NEURO-CINÉTICO-FUNCIONAIS COM COMPLICAÇÕES SISTÊMICAS - PEDIATRICO',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS DESORDENS DO DESENVOLVIMENTO NEURO MOTOR',
                    'ATENDIMENTO FISIOTERAPÊUTICO NAS DESORDENS DO DESENVOLVIMENTO NEURO MOTOR - Pediátrico',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE C/ COMPROMETIMENTO COGNITIVO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE NO PRÉ/PÓS-OPERATÓRIO DE NEUROCIRURGIA',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE MÉDIO QUEIMADO',
                    'ATENDIMENTO FISIOTERAPÊUTICO EM PACIENTE COM SEQÜELAS POR QUEIMADURAS (MÉDIO E GRANDE QUEIMADOS)'
                )
                THEN 'FISIOTERAPIA ATENDIMENTO (SESSÕES)'
    
                WHEN procedimento IN (
                    'CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA ATENÇÃO ESPECIALIZADA (EXCETO MÉDICO) Adulto - Fisioterapia',
                    'CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA ATENÇÃO ESPECIALIZADA (EXCETO MÉDICO) Pediatrico - Fisioterapia'
                )
                THEN 'FISIOTERAPIA CONSULTA'
    
                WHEN procedimento IN (
                    'ARMAÇÃO PARA ÓCULOS INFANTIL',
                    'ARMAÇÃO PARA ÓCULOS ADULTO',
                    'LENTE COMUM (PAR), GRADUAÇÃO INDETERMINADA. ESFERICA:+4.00/-4.00 CIL: 0.00 A -2.00',
                    'LENTE ESPECIAL (PAR), GRADUAÇÃO INDETERMINADA. ESF: +6.00/-8.00 CIL: 0.00A -3.00',
                    'LENTE BIFOCAL BAIXO (PAR).ESF:-4.00/+7.00 CILÍNDRICO: ATÉ 4.00',
                    'LENTE BIFOCAL MÉDIO (PAR).ESF:-4.00/+7.00 CILÍNDRICO: ATÉ 4.00',
                    'LENTE MULTIFOCAL BAIXO (PAR). ESFERICA: -4.00/+7.00 CILÍNDRICO: ATÉ 4.00',
                    'LENTE MULTIFOCAL MÉDIO (PAR). ESFERICA: -6.00/+7.00 CIL: ATÉ 4.00',
                    'LENTE MULTIFOCAL ALTO (PAR). ESFERICA: -10.00/+6.00 CIL: ATÉ -4.00',
                    'LENTE ESPECIAL ALTO ÍNDICE (PAR). ESFERICA: +/-20.00 CIL: ATÉ -8.00',
                    'LENTE COMUM (PAR), GRADUAÇÃO INDETERMINADA. ESFERICA:+4.00/-4.00 CIL: 0.00 A -2.00 COM LENTE FOTOSSENSÍVEL',
                    'LENTE ESPECIAL (PAR), GRADUAÇÃO INDETERMINADA. ESF: +6.00/- 8.00 CIL: 0.00A -3.00 COM LENTE E FOTOSSENSÍVEL.',
                    'LENTE BIFOCAL BAIXO (PAR). ESF: -4.00/+7.00 CILÍNDRICO: ATÉ 4.00 COM LENTE FOTOSSENSÍVEL',
                    'LENTE BIFOCAL ALTO (PAR). ESF: -10.00/+7.00 CILÍNDRICO: ATÉ - 4.00'
                )
                THEN 'FORNECIMENTO DE ÓCULOS DE GRAU, INCLUINDO A ARMAÇÃO E AS LENTES CORRETIVAS CONFORME INDICAÇÃO MEDICA'
    
                WHEN procedimento IN (
                    'CONSULTA - GINECOLOGIA E COLETA MATERIAL EXAME CITOPATOLOGICO COLO UTERINO C/ RESULTADO E CONSULTA DE RETORNO GINECOLOGIA'
                )
                THEN 'GINECOLOGIA E COLETA MATERIAL EXAME CITOPATOLOGICO COLO UTERINO C/ RESULTADO E CONSULTA DE RETORNO GINECOLOGIA'
    
                WHEN procedimento IN (
                    'MONITORAMENTO PELO SISTEMA HOLTER 24 HS (3 CANAIS)'
                )
                THEN 'HOLTER 24H(3 CANAIS)'
    
                WHEN procedimento IN (
                    'MONITORIZACAO AMBULATORIAL DE PRESSAO ARTERIAL (M.A.P.A)'
                )
                THEN 'M.A.P.A.'
    
                WHEN procedimento IN (
                    'MAMOGRAFIA BILATERAL PARA RASTREAMENTO - Registro no Sistema SISMAMA',
                    'MAMOGRAFIA'
                )
                THEN 'MAMOGRAFIA'
                
                WHEN procedimento IN (
                    'POLISSONOGRAFIA'
                )
                THEN 'POLISSONOGRAFIA'
    
                WHEN procedimento IN (
                    'POTENCIAL EVOCADO AUDITIVO PARA TRIAGEM AUDITIVA (TESTE DA ORELHINHA)'
                )
                THEN 'POTENCIAL EVOCADO AUDITIVO PARA TRIAGEM AUDITIVA (TESTE DA ORELHINHA)'
    
                WHEN procedimento IN (
                    'CERATOMETRIA',
                    'BIOMETRIA ULTRASSÔNICA (MONOCULAR)',
                    'MAPEAMENTO DE RETINA',
                    'MICROSCOPIA ESPECULAR DE CORNEA AMBOS OS OLHOS',
                    'POTENCIAL DE ACUIDADE VISUAL',
                    'RETINOGRAFIA COLORIDA BINOCULAR',
                    'RETINOGRAFIA FLUORESCENTE BINOCULAR',
                    'CAMPIMETRIA COMPUTADORIZADA OU MANUAL COM GRÁFICO',
                    'PAQUIMETRIA ULTRASSÔNICA AMBOS OS OLHOS',
                    'TOPOGRAFIA COMPUTADORIZADA DE CÓRNEA/ CERATOSCOPIA'
                )
                THEN 'PROC. OFTALMOLOGICO'
    
                WHEN procedimento IN (
                    'PRÓTESE PARCIAL MANDIBULAR REMOVÍVEL',
                    'PRÓTESE PARCIAL MAXILAR REMOVÍVEL',
                    'PRÓTESE TOTAL MANDIBULAR',
                    'PRÓTESE TOTAL MAXILAR',
                    'PRÓTESE CORONÁRIAS/INTRARRADICULARES FIXAS/ ADESIVAS (POR ELEMENTO)'
                )
                THEN 'PROTESE DENTARIA'
    
                WHEN procedimento IN (
                    'RADIOGRAFIA BILATERAL DE ORBITAS (PA + OBLIQUAS + HIRTZ)',
                    'RADIOGRAFIA DE ARCADA ZIGOMATICO-MALAR (AP+ OBLIQUAS)',
                    'RADIOGRAFIA DE ARTICULACAO TEMPORO-MANDIBULAR BILATERAL',
                    'RADIOGRAFIA DE CAVUM (LATERAL + HIRTZ)',
                    'RADIOGRAFIA DE CRANIO (PA + LATERAL + OBLÍQUA / BRETTON + HIRTZ)',
                    'RADIOGRAFIA DE CRANIO (PA + LATERAL)',
                    'RADIOGRAFIA DE LARINGE',
                    'RADIOGRAFIA DE MASTOIDE / ROCHEDOS (BILATERAL)',
                    'RADIOGRAFIA DE MAXILAR (PA + OBLIQUA)',
                    'RADIOGRAFIA DE OSSOS DA FACE (MN + LATERAL + HIRTZ)',
                    'RADIOGRAFIA DE REGIAO ORBITARIA (LOCALIZACAO DE CORPO ESTRANHO)',
                    'RADIOGRAFIA DE SEIOS DA FACE (FN + MN + LATERAL + HIRTZ)',
                    'RADIOGRAFIA DE SELA TURSICA (PA + LATERAL + BRETTON)',
                    'RADIOGRAFIA DE COLUNA CERVICAL (AP + LATERAL + TO + OBLIQUAS)',
                    'RADIOGRAFIA DE COLUNA CERVICAL (AP + LATERAL + TO / FLEXAO)',
                    'RADIOGRAFIA DE COLUNA CERVICAL FUNCIONAL / DINAMICA',
                    'RADIOGRAFIA DE COLUNA LOMBO-SACRA',
                    'RADIOGRAFIA DE COLUNA LOMBO-SACRA (C/ OBLIQUAS)',
                    'RADIOGRAFIA DE COLUNA LOMBO-SACRA FUNCIONAL / DINAMICA',
                    'RADIOGRAFIA DE COLUNA TORACICA (AP + LATERAL)',
                    'RADIOGRAFIA DE COLUNA TORACO-LOMBAR',
                    'RADIOGRAFIA DE COLUNA TORACO-LOMBAR DINAMICA',
                    'RADIOGRAFIA DE REGIAO SACRO-COCCIGEA',
                    'RADIOGRAFIA DE CORACAO E VASOS DA BASE (PA + LATERAL + OBLIQUA)',
                    'RADIOGRAFIA DE CORACAO E VASOS DA BASE (PA + LATERAL)',
                    'RADIOGRAFIA DE COSTELAS (POR HEMITORAX)',
                    'RADIOGRAFIA DE ESOFAGO',
                    'RADIOGRAFIA DE ESTERNO',
                    'RADIOGRAFIA DE MEDIASTINO (PA E PERFIL)',
                    'RADIOGRAFIA DE PNEUMOMEDIASTINO',
                    'RADIOGRAFIA DE TORAX (APICO-LORDORTICA)',
                    'RADIOGRAFIA DE TÓRAX (PA + INSPIRAÇÃO + EXPIRAÇÃO + LATERAL)',
                    'RADIOGRAFIA DE TORAX (PA + LATERAL + OBLIQUA)',
                    'RADIOGRAFIA DE TORAX (PA + PERFIL)',
                    'RADIOGRAFIA DE TORAX (PA PADRAO OIT)',
                    'RADIOGRAFIA DE TORAX (PA)',
                    'RADIOGRAFIA DE ANTEBRACO',
                    'RADIOGRAFIA DE ARTICULACAO ACROMIO-CLAVICULAR',
                    'RADIOGRAFIA DE ARTICULACAO ESCAPULO-UMERAL',
                    'RADIOGRAFIA DE ARTICULACAO ESTERNO-CLAVICULAR',
                    'RADIOGRAFIA DE BRACO',
                    'RADIOGRAFIA DE CLAVICULA',
                    'RADIOGRAFIA DE COTOVELO',
                    'RADIOGRAFIA DE DEDOS DA MAO',
                    'RADIOGRAFIA DE MAO',
                    'RADIOGRAFIA DE MAO E PUNHO (P/ DETERMINACAO DE IDADE OSSEA)',
                    'RADIOGRAFIA DE ESCAPULA/OMBRO (TRES POSICOES)',
                    'RADIOGRAFIA DE PUNHO (AP + LATERAL + OBLIQUA)',
                    'RADIOGRAFIA DE ABDOMEN (AP + LATERAL / LOCALIZADA)',
                    'RADIOGRAFIA DE ABDOMEN AGUDO (MINIMO DE 3 INCIDENCIAS)',
                    'RADIOGRAFIA DE ABDOMEN SIMPLES (AP)',
                    'RADIOGRAFIA DE ESTOMAGO E DUODENO',
                    'RADIOGRAFIA DE INTESTINO DELGADO (TRANSITO)',
                    'RADIOGRAFIA DE ARTICULACAO COXO-FEMORAL',
                    'RADIOGRAFIA DE ARTICULACAO SACRO-ILIACA',
                    'RADIOGRAFIA DE ARTICULACAO TIBIO-TARSICA',
                    'RADIOGRAFIA DE BACIA',
                    'RADIOGRAFIA DE CALCANEO',
                    'RADIOGRAFIA DE JOELHO (AP + LATERAL)',
                    'RADIOGRAFIA DE JOELHO OU PATELA (AP + LATERAL + AXIAL)',
                    'RADIOGRAFIA DE JOELHO OU PATELA (AP + LATERAL + OBLIQUA + 3 AXIAIS)',
                    'RADIOGRAFIA DE PE / DEDOS DO PE',
                    'RADIOGRAFIA DE PERNA',
                    'BRONCOGRAFIA UNILATERAL',
                    'DUCTOGRAFIA (POR MAMA)',
                    'COLANGIOGRAFIA PER-OPERATORIA',
                    'COLANGIOGRAFIA POS-OPERATORIA',
                    'HISTEROSSALPINGOGRAFIA'
                )
                THEN 'RADIOGRAFIA'
    
                WHEN procedimento IN (
                    'RESSONANCIA MAGNETICA DE ARTICULACAO TEMPORO-MANDIBULAR (BILATERAL)',
                    'RESSONANCIA MAGNETICA DE COLUNA CERVICAL/PESCOÇO',
                    'RESSONANCIA MAGNETICA DE COLUNA LOMBO-SACRA',
                    'RESSONANCIA MAGNETICA DE COLUNA TORACICA',
                    'RESSONÂNCIA MAGNÉTICA DA MAMA BILATERAL',
                    'RESSONANCIA MAGNETICA DE CRANIO',
                    'RESSONANCIA MAGNETICA DE SELA TURCICA',
                    'RESSONANCIA MAGNETICA CARDÍACA – MORFOLÓGICA E FUNCIONAL',
                    'RESSONANCIA MAGNETICA CARDÍACA – MORFOLÓGICA E FUNCIONAL E PERFUSÃO',
                    'RESSONANCIA MAGNETICA DE MEMBRO SUPERIOR (UNILATERAL)',
                    'RESSONANCIA MAGNETICA DE TORAX',
                    'RESSONANCIA MAGNETICA DE ABDOMEN SUPERIOR',
                    'RESSONANCIA MAGNETICA DE BACIA / PELVE / ABDOMEN INFERIOR',
                    'RESSONANCIA DE PROSTATA',
                    'RESSONANCIA MAGNETICA DE MEMBRO INFERIOR (UNILATERAL)',
                    'RESSONANCIA MAGNETICA DE VIAS BILIARES/COLANGIORRESSONANCIA',
                    'ANGIORESSONANCIA CEREBRAL'
                )
                THEN 'RESSONANCIA MAGNETICA'
                
                WHEN procedimento IN (
                    'TESTE DE ESFORCO / TESTE ERGOMETRICO'
                )
                THEN 'TESTE DE ESFORCO / TESTE ERGOMETRICO'
    
                WHEN procedimento IN (
                    'TOMOGRAFIA COMPUTADORIZADA DE COLUNA CERVICAL',
                    'TOMOGRAFIA COMPUTADORIZADA DE COLUNA LOMBO-SACRA',
                    'TOMOGRAFIA COMPUTADORIZADA DE COLUNA TORACICA',
                    'TOMOGRAFIA COMPUTADORIZADA DE FACE / SEIOS DA FACE / ARTICULACOES TEMPORO-MANDIBULARES',
                    'TOMOGRAFIA COMPUTADORIZADA DO PESCOCO',
                    'TOMOGRAFIA COMPUTADORIZADA DE SELA TURCICA',
                    'TOMOGRAFIA COMPUTADORIZADA DO CRANIO',
                    'TOMOGRAFIA COMPUTADORIZADA DE CRANIO',
                    'TOMOGRAFIA COMPUTADORIZADA DE SEGMENTOS APENDICULARES - (BRACO, ANTEBRAÇO, MÃO, COXA, PERNA, PÉ)',
                    'TOMOGRAFIA COMPUTADORIZADA DE TORAX',
                    'TOMOGRAFIA COMPUTADORIZADA DE ABDOMEN SUPERIOR',
                    'TOMOGRAFIA COMPUTADORIZADA DE ARTICULACOES DE MEMBRO INFERIOR',
                    'TOMOGRAFIA COMPUTADORIZADA DE PELVE / BACIA / ABDOMEN INFERIOR',
                    'TOMOGRAFIA DE COERÊNCIA ÓPTICA',
                    'TOMOGRAFIA COMPUTADORIZADA DE ARTICULACOES DE MEMBRO SUPERIOR'
                )
                THEN 'TOMOGRAFIA'
    
                WHEN procedimento IN (
                    'ULTRASSONOGRAFIA DE ABDÔMEN SUPERIOR',
                    'ULTRASSONOGRAFIA DE ABDOMEN TOTAL',
                    'ULTRASSONOGRAFIA DE APARELHO URINÁRIO',
                    'ULTRASSONOGRAFIA DE PARTES MOLES',
                    'ULTRASSONOGRAFIA DE ARTICULACAO',
                    'ULTRASSONOGRAFIA DE BOLSA ESCROTAL',
                    'ULTRASSONOGRAFIA DE GLOBO OCULAR / ORBITA (MONOCULAR)',
                    'ULTRASSONOGRAFIA MAMARIA BILATERAL',
                    'ULTRASSONOGRAFIA DE ARTICULACAO E PARTES MOLES',
                    'ULTRASSONOGRAFIA DE PROSTATA POR VIA ABDOMINAL COM MEDIDA DE RESIDUO POS MICCIONAL',
                    'ULTRASSONOGRAFIA DE PROSTATA (VIA TRANSRETAL)',
                    'ULTRASSONOGRAFIA DE TIREOIDE',
                    'ULTRASSONOGRAFIA OBSTETRICA',
                    'ULTRASSONOGRAFIA OBSTÉTRICA MORFOLOGICA POR FETO',
                    'ULTRASSONOGRAFIA PELVICA (GINECOLOGICA)',
                    'ULTRASSONOGRAFIA TRANSFONTANELA',
                    'ULTRASSONOGRAFIA TRANSVAGINAL',
                    'ECOCARDIOGRAFIA (FETAL)',
                    'ULTRASSONOGRAFIA DE PROSTATA POR VIA ABDOMINAL'
                )
                THEN 'ULTRASSONOGRAFIA'
    
                WHEN procedimento IN (
                    'ECOCARDIOGRAFIA DOPPLER COLORIDO',
                    'ULTRASSONOGRAFIA BOLSA ESCROTAL COM DOPPLER',
                    'ULTRASSONOGRAFIA DOPPLER DE ARTERIAS POPLITEA',
                    'ULTRASSONOGRAFIA DOPPLER DE ARTERIAS RENAIS',
                    'ULTRASSONOGRAFIA DOPPLER DE PORTA HEPATICA',
                    'ULTRASSONOGRAFIA DOPPLER DE VEIAS ILIACAS',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO INFERIOR ARTERIAL',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO INFERIOR VENOSO',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO SUPERIOR ARTERIAL',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - MEMBRO SUPERIOR VENOSO',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO TRANSVAGINAL',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE PENIS',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE TIREOIDE',
                    'ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS - CAROTIDAS E VERTEBRAIS',
                    'ULTRASSONOGRAFIA DOPPLER DE FLUXO OBSTETRICO',
                    'ULTRASSONOGRAFIA OBSTETRICA C/ DOPPLER COLORIDO E PULSADO'
                )
                THEN 'US DOPPLER'
    
                WHEN procedimento IN (
                    'TESTES VESTIBULARES / OTONEUROLOGICOS'
                )
                THEN 'VECTOELETRONISTAGMOGRAFIA/ testes vestibulares'
    
                WHEN procedimento IN (
                    'HISTEROSCOPIA (VIDEO HISTEROSCOPIA DIAGNÓSTICA) (SEM  SEDAÇÃO)',
                    'HISTEROSCOPIA (VIDEO HISTEROSCOPIA DIAGNOSTICA) COM BIOPSIA E RESULTADO DA BIÓPSIA (SEM  SEDAÇÃO)'
                )
                THEN 'VIDEOHISTEROSCOPIA'
    
                WHEN procedimento IN (
                    'VIDEOLARINGOSCOPIA'
                )
                THEN 'VIDEOLARINGOSCOPIA'
    
                ELSE 'outro'
            END AS grupo,
            procedimento,
            cidade
    
        FROM agendamentos
    
        WHERE situacao = 'VALIDADO'
    )
    SELECT
        procedimento,
        COUNT(*) AS quantidade,
        concat_ws(', ', collect_set(cidade)) AS cidades
    FROM dados
    WHERE grupo = 'outro'
    GROUP BY procedimento
    ORDER BY quantidade DESC;

""").show()

+------------+----------+-------+
|procedimento|quantidade|cidades|
+------------+----------+-------+
+------------+----------+-------+



In [ ]:
import socket
print(socket.gethostbyname("minio"))